<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 8 — A Worked Example: Yelp Businesses

One question, taken from a raw file to a conclusion you could defend:

> **Do more expensive restaurants get better ratings?**

It sounds like one `groupby`. It is one `groupby` — at the very end, after
five steps that decide whether its answer means anything.

```text
 1 load  ──▶  2 inspect  ──▶  3 clean  ──▶  4 whole-column  ──▶  5 group  ──▶  6 the blanks  ──▶  7 conclude
                                               work
```

This is the same path Homework 7 takes through a different file. Where
notebook `01_Thinking_in_Tables` introduced each idea on eight rows, this one
uses them on 150,346.

**The data.** The Yelp academic dataset, which you streamed line by line in
Homework 4. This notebook uses only the business file: one record per
business, with its location, star rating, review count and a bundle of
attributes.

**Before you start**

- Run this notebook **on ARC**. The file lives in `/data/public/yelp/` and is
  reachable only from there.
- Set the kernel (upper right corner) to **Conda Python 3.12**.
- Run the cells in order with **Shift + Enter**.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.options.display.float_format = "{:,.3f}".format
pd.options.display.max_columns = 20
pd.options.display.width = 160

BUSINESS = Path("/data/public/yelp/yelp_academic_dataset_business.json")

---

## Step 1 — Load

The file is **JSON Lines**: one JSON object per line, not one big list.
In Homework 4 you read such a file with `json.loads` on each line.
`pd.read_json(..., lines=True)` does that for the whole file and returns a
table.

In [ ]:
with open(BUSINESS, encoding="utf-8") as handle:
    print(handle.readline()[:600])

In [ ]:
business = pd.read_json(BUSINESS, lines=True)
business.shape

150,346 rows and 14 columns, in a couple of seconds. The file is 119 MB; it
fits in memory comfortably. (The review file from Homework 4 is 5 GB and does
not. That is when streaming is still the right tool.)

| File looks like | Read it with |
|---|---|
| `[{"a": 1}, {"a": 2}]` — one list of records | `pd.read_json(path)` |
| `{"a": 1}` then `{"a": 2}` on the next line | `pd.read_json(path, lines=True)` |
| a header line, then comma-separated rows | `pd.read_csv(path)` |

---

## Step 2 — Inspect

The five lines from notebook 01, before any question is asked.

In [ ]:
business.dtypes

In [ ]:
business.sample(4, random_state=1).T        # .T turns rows into columns: easier to read wide records

**What is one row?** One business. Check that the identifier is unique, as you
did for every table in Homework 6:

In [ ]:
business["business_id"].is_unique

In [ ]:
business.isna().sum()

Three columns have missing values: `attributes`, `categories` and `hours`.
Everything else is complete — or looks it. Keep that thought.

In [ ]:
business[["stars", "review_count", "is_open", "latitude", "longitude"]].describe()

Read the table, do not just glance at it.

- `stars` runs from 1 to 5 with a mean of about 3.6. Believable.
- `review_count` has a median of 15 and a maximum of 7,568. Half of all
  businesses have 15 reviews or fewer. That will matter: a 5-star rating from
  five reviews is not the same evidence as one from five hundred.
- `is_open` is 0 or 1, and its **mean is 0.796**. The mean of a 0/1 column is
  a share, so 79.6 % of the businesses are still open.

In [ ]:
business["state"].value_counts()

27 states, and the counts fall off a cliff. Pennsylvania has 34,039
businesses; Texas has 4. This is not a sample of the country. It is a handful
of metropolitan areas, with a few stray records from elsewhere. **Any
"per state" result for Texas would be an average of four businesses.**

### Blanks that `isna()` did not report

In [ ]:
for column in ["address", "postal_code", "city"]:
    print(f"{column:12} empty text: {(business[column] == '').sum():>6,}")

`address` reported zero missing values, and 5,127 of them are the empty text
`""`. A blank in disguise, exactly as in notebook 01.

In [ ]:
print("different city names:              ", business["city"].nunique())
print("after stripping spaces, lowercasing:",
      business["city"].str.strip().str.lower().nunique())
business.loc[business["city"].str.lower().str.strip() == "philadelphia",
             "city"].value_counts().head(6)

About 180 "different" cities are the same city typed differently. Group by
`city` as it stands and Philadelphia is split across several rows.

**What inspection found, before any analysis:**

| Finding | Consequence |
|---|---|
| Half the businesses have ≤ 15 reviews | a rating alone is weak evidence |
| 13 of 27 states have under 1,000 businesses | do not compare states without a size threshold |
| `address` hides 5,127 blanks as `""` | `isna()` under-reports |
| city names are inconsistent | clean before grouping by city |
| `attributes`, `categories`, `hours` are missing in places | Step 6 |

**✏️ Try it:** How many businesses have `review_count` of at least 100? What share is that? (One comparison, then `.sum()` and `.mean()`.)

---

## Step 3 — Clean, and keep what the question needs

The question is about **restaurants** and their **price**. Neither is a column.

### Which businesses are restaurants?

`categories` is one piece of text listing every category a business belongs
to:

In [ ]:
business["categories"].head(4).tolist()

In [ ]:
# .str methods work on a whole column of text.  na=False: a business with no
# categories at all is not a restaurant.
business["is_restaurant"] = business["categories"].str.contains("Restaurants", na=False)

print("restaurants:", business["is_restaurant"].sum())
print("share:      ", round(business["is_restaurant"].mean(), 3))

Without `na=False` the 103 businesses with no categories would get a missing
value in a column that should be True/False, and using it to choose rows would
fail. Deciding that "no categories listed" means "not a restaurant" is a
decision about a blank. It is a small one, and it is now written down.

### Where is the price?

`attributes` holds a **dictionary** in every cell: the nested part of the JSON
record.

In [ ]:
business["attributes"].iloc[3]

In [ ]:
# .str.get(key) looks a key up in every dictionary of the column.
price_text = business["attributes"].str.get("RestaurantsPriceRange2")
price_text.value_counts(dropna=False)

Three things to see:

1. The price is 1 to 4 (one to four dollar signs on the website), stored as
   **text**: `'2'`, not `2`.
2. `None` appears 65,032 times: the business has no price recorded.
3. `'None'` — the *word*, in quotes — appears 34 times. A blank that somebody
   wrote down as text. `isna()` would not catch it.

`pd.to_numeric(..., errors="coerce")` settles all three at once: text that is
a number becomes a number, and everything else becomes a real missing value.

In [ ]:
business["price"] = pd.to_numeric(price_text, errors="coerce")
business["price"].value_counts(dropna=False).sort_index()

### Keep the rows and columns the question needs

In [ ]:
restaurants = business.loc[
    business["is_restaurant"],
    ["business_id", "name", "city", "state", "stars", "review_count",
     "is_open", "price", "hours"],
].copy()

restaurants["city"] = restaurants["city"].str.strip().str.title()
restaurants.shape

The `.copy()` matters. Without it `restaurants` would be a view onto part of
`business`, and adding a column to it would raise the
`SettingWithCopyWarning` that notebook 03 explains.

---

## Step 4 — Whole-column work

Three new columns, each one expression over a whole column. No loop.

In [ ]:
# A True/False column from a comparison.
restaurants["well_rated"] = restaurants["stars"] >= 4.0

# A label that depends on conditions.  np.select is np.where with more than
# two outcomes: the first condition that is True wins.
reviews = restaurants["review_count"]
restaurants["evidence"] = np.select(
    [reviews < 20, reviews < 200],
    ["thin", "fair"],
    default="solid",
)

# A flag for a missing value, kept as its own column.
restaurants["has_hours"] = restaurants["hours"].notna()

restaurants[["name", "stars", "review_count", "well_rated", "evidence",
             "has_hours"]].head(6)

In [ ]:
restaurants["evidence"].value_counts(normalize=True)

`value_counts(normalize=True)` gives shares instead of counts. A third of the
restaurants have fewer than 20 reviews behind their rating.

**✏️ Try it:** What share of restaurants is `well_rated`? And among those with `evidence == "solid"`? (Choose rows with a condition, then take the mean of the True/False column.)

---

## Step 5 — Split, apply, combine

The question at last: rating per price level.

In [ ]:
by_price = restaurants.groupby("price", as_index=False).agg(
    restaurants=("business_id", "size"),
    mean_stars=("stars", "mean"),
    well_rated=("well_rated", "mean"),
    median_reviews=("review_count", "median"),
    still_open=("is_open", "mean"),
)
by_price

Read it column by column.

- `mean_stars` climbs from about 3.30 at price 1 to about 3.68 at price 3,
  then stops climbing.
- `restaurants` says how much each row can be trusted: price 4 is 194
  restaurants out of more than 50,000.
- `median_reviews` is 25 at level 1 and 66 at level 2. Cheap places are less
  reviewed, so their ratings rest on less evidence.

**Does the table add up?** It should account for every restaurant.

In [ ]:
print("restaurants in the table:", by_price["restaurants"].sum())
print("restaurants altogether:  ", len(restaurants))
print("not in the table:        ", len(restaurants) - by_price["restaurants"].sum())

Nearly 7,800 restaurants are missing from the result. They have no `price`,
and `groupby` drops rows whose key is missing. Put them back:

In [ ]:
by_price = restaurants.groupby("price", as_index=False, dropna=False).agg(
    restaurants=("business_id", "size"),
    mean_stars=("stars", "mean"),
    well_rated=("well_rated", "mean"),
    median_reviews=("review_count", "median"),
    still_open=("is_open", "mean"),
)
by_price

The restaurants with no price are a group of their own, and not a small one.
Step 6 looks at who they are.

### Compared with what? — `transform`

A 4-star restaurant in a city where the average is 4.2 is below par. To
compare each restaurant with **its own city**, the city's average has to sit
beside every row. That is `transform`.

In [ ]:
city_mean = restaurants.groupby("city")["stars"].transform("mean")
city_size = restaurants.groupby("city")["stars"].transform("size")

restaurants["vs_city"] = restaurants["stars"] - city_mean
restaurants["city_size"] = city_size

restaurants[["name", "city", "stars", "vs_city", "city_size"]].head(6)

In [ ]:
# The price question again, now relative to each restaurant's own city, and
# only in cities with at least 200 restaurants.
big_cities = restaurants[restaurants["city_size"] >= 200]

big_cities.groupby("price").agg(
    restaurants=("business_id", "size"),
    stars_vs_city=("vs_city", "mean"),
)

The pattern survives. Within the same city, cheaper restaurants sit below
their city's average and more expensive ones above it. So the result in the
first table was not just "expensive restaurants happen to be in cities that
rate generously".

### Only the groups that qualify — a size threshold

Inspection warned that some states are tiny. Compare states only where there
is enough to compare.

In [ ]:
by_state = restaurants.groupby("state", as_index=False).agg(
    restaurants=("business_id", "size"),
    mean_stars=("stars", "mean"),
    still_open=("is_open", "mean"),
)
print(len(by_state), "states")
by_state[by_state["restaurants"] >= 1000].sort_values("mean_stars", ascending=False)

**✏️ Try it:** Which **city** with at least 500 restaurants has the highest mean rating? Build the per-city table with `agg`, keep the big ones, sort.

---

## Step 6 — What do the blanks mean?

Two columns have blanks that matter here: `price` and `hours`.

### `hours`: is the blank telling you something?

In [ ]:
restaurants.groupby("has_hours").agg(
    restaurants=("business_id", "size"),
    still_open=("is_open", "mean"),
    mean_stars=("stars", "mean"),
    median_reviews=("review_count", "median"),
)

Restaurants with no opening hours listed are far more likely to be **closed
for good**, and they have far fewer reviews. The blank is not random. A
missing value here is itself a piece of information about the business.

That is the general lesson, and it is why "just drop the rows with missing
values" is a decision and not a cleanup step: **the rows you drop are often
different from the rows you keep.**

### `price`: three ways to treat the blank

In [ ]:
print(f"restaurants with no price: {restaurants['price'].isna().sum():,} "
      f"({restaurants['price'].isna().mean():.1%})")

restaurants.assign(has_price=restaurants["price"].notna()).groupby("has_price").agg(
    restaurants=("business_id", "size"),
    mean_stars=("stars", "mean"),
    median_reviews=("review_count", "median"),
    still_open=("is_open", "mean"),
)

A different story from `hours`. Restaurants without a price have far fewer
reviews, but they are rated **higher** and are **more** often still open. They
look like small or new places that nobody has filled in a price for.

So the two blanks mean different things, in the same table. You could not
have guessed either from the column name.

What is the average price level of a restaurant?

In [ ]:
price = restaurants["price"]
city_median = restaurants.groupby("city")["price"].transform("median")

policies = pd.DataFrame({
    "policy": ["leave blanks out", "blank means cheapest (1)",
               "fill with the city's median"],
    "mean_price": [price.mean(),
                   price.fillna(1).mean(),
                   price.fillna(city_median).mean()],
    "rows_used": [price.notna().sum(),
                  price.fillna(1).notna().sum(),
                  price.fillna(city_median).notna().sum()],
})
policies

The three answers are close together this time, because only about 15 % of the
values are missing and the fills are near the middle. In Homework 7 the same
three policies on a column that is half empty give 17.1, 8.5 and 18.4.

**How much a policy matters depends on how much is missing and on how
different the blank rows are.** You cannot know either without looking.

Note `rows_used` in the third row. A few restaurants are in cities where no
restaurant has a price, so there was no median to fill with, and they stayed
blank. That is the right outcome.

---

## Step 7 — Conclude, and say what the conclusion rests on

> **Do more expensive restaurants get better ratings?**

In [ ]:
solid = restaurants[(restaurants["evidence"] != "thin") & restaurants["price"].notna()]

answer = solid.groupby("price", as_index=False).agg(
    restaurants=("business_id", "size"),
    mean_stars=("stars", "mean"),
    well_rated=("well_rated", "mean"),
)
answer

Compare this with the first table in Step 5. There, level 4 was no better
rated than level 3. Here, with the thinly reviewed restaurants set aside, it
is clearly the highest. The earlier flat spot came from ratings that rested on
a handful of reviews.

An honest write-up of this table has four parts, and only the first is the
number.

1. **The finding.** Among restaurants with at least 20 reviews and a recorded
   price, the mean rating rises with every price level, from about 3.45 at
   level 1 to about 3.94 at level 4: half a star in all.
2. **What it rests on.** The number of restaurants in each row, shown in the
   table. Level 4 is a very small group.
3. **What was left out, and why.** Restaurants with fewer than 20 reviews, and
   the roughly 15 % with no price recorded. Those without a price are less
   reviewed, rated higher and more often open, so they are not a random
   omission.
4. **What it does not show.** That raising prices raises ratings. Expensive
   restaurants differ from cheap ones in many ways at once. This is a pattern
   in a table, not a cause.

### Saving a result

A result table goes to a file the same way a data table comes from one. JSON
as a list of records keeps the column names with every row:

In [ ]:
print(answer.to_json(orient="records", indent=2))

`answer.to_json("price_and_rating.json", orient="records")` writes that text to
a file, and `pd.read_json("price_and_rating.json")` reads it back.
`answer.to_csv("price_and_rating.csv", index=False)` does the same as CSV.

---

## Exercises

All on `business` and `restaurants` as built above.

### 🧩 Exercise 1 — Inspect

`postal_code` is text. How many businesses have an empty one? In how many
different states do they occur? Are they more or less often open than
businesses with a postal code?

### 🧩 Exercise 2 — A column from nested data

`attributes` has a key `"WiFi"`. Pull it out with `.str.get("WiFi")` and look
at `value_counts(dropna=False)`. The same answer is spelled several ways.

Build a clean column `wifi` with the values `"free"`, `"paid"`, `"no"` and
missing. `.str.replace` and `.str.strip` will do it, or `np.select` with
`.str.contains`.

### 🧩 Exercise 3 — Split, apply, combine

For restaurants only: one row per `evidence` level (`thin`, `fair`, `solid`)
with the number of restaurants, the mean rating, and the **standard deviation**
of the rating (`"std"`).

Which level has the most spread-out ratings? Why would that be?

### 🧩 Exercise 4 — Transform

Add a column `reviews_vs_city`: each restaurant's `review_count` divided by
the **median** review count of restaurants in its city. Which restaurant has
the largest value, and what does a value of 10 mean in words?

### 🧩 Exercise 5 — What does the blank mean?

Compute the share of restaurants that are still open, three ways: over all
restaurants; only those with `hours`; only those with `price`. Then explain in
two sentences why the three differ, and which one you would quote.

---

## Summary

| Step | What was done | The idea from notebook 01 |
|---|---|---|
| 1 Load | `pd.read_json(path, lines=True)` | the table is the central structure |
| 2 Inspect | `dtypes`, `sample`, `isna().sum()`, `describe`, `value_counts` | inspect before you conclude |
| 3 Clean | `.str.contains`, `.str.get`, `pd.to_numeric(errors="coerce")` | missing values in disguise |
| 4 Whole-column work | comparisons, `np.select`, `notna` | vectorized thinking |
| 5 Group | `agg`, `transform`, a size threshold, `dropna=False` | split, apply, combine |
| 6 The blanks | compare rows with and without; three fill policies | a missing value is a question |
| 7 Conclude | the number, what it rests on, what was left out, what it does not show | all of them |

Homework 7 walks the same seven steps through the Forest Service
invasive-plants file. `03_Data_Analysis_Reference` has every method used here,
with its options.